In [5]:
#cargamos las tres novelas y aplicamos segmentacion y tokenizacion

from pathlib import Path
import re
import pandas as pd
import random

CSV_PATH = Path("gutenberg_novels_dataset.csv")

df = pd.read_csv(CSV_PATH)

#revisamos los libros disponibles
print("Libros encontrados en el dataset")
print(df[["title", "author"]])

#seleccionamos las tres novelas del laboratorio
novelas = df[df["title"].isin([
    "Pride and Prejudice",
    "Frankenstein",
    "Dracula"
])].copy()

if len(novelas) != 3:
    raise ValueError("No se encontraron exactamente las tres novelas esperadas")

#tokenizamos cada novela por separado
patron_palabras = re.compile(
    r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+(?:[-'][A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+)*|\d+(?:[.,]\d+)*",
    re.UNICODE
)

#procesamos cada novela y guardamos los resultados en un diccionario
novelas_procesadas = {}

for _, fila in novelas.iterrows():
    texto = str(fila["text"])
    texto = re.sub(r"\s+", " ", texto).strip()

    #hacemos la segmentacion en oraciones
    oraciones_raw = re.split(r"(?<=[.!?])\s+", texto)

    #tokenizamos cada oracion sin quitar stopwords ni lematizar
    oraciones_tokenizadas = [
        patron_palabras.findall(oracion)
        for oracion in oraciones_raw
    ]

    oraciones_tokenizadas = [
        oracion
        for oracion in oraciones_tokenizadas
        if len(oracion) > 0
    ]

    novelas_procesadas[fila["title"]] = {
        "autor": fila["author"],
        "texto": texto,
        "oraciones": oraciones_tokenizadas
    }

    print(f"\nLibro {fila['title']}")
    print(f"Autor {fila['author']}")
    print(f"Total de oraciones {len(oraciones_tokenizadas):,}")
    print(f"Total de tokens {sum(len(oracion) for oracion in oraciones_tokenizadas):,}")

Libros encontrados en el dataset
                 title        author
0  Pride and Prejudice   Jane Austen
1         Frankenstein  Mary Shelley
2              Dracula   Bram Stoker

Libro Pride and Prejudice
Autor Jane Austen
Total de oraciones 5,943
Total de tokens 128,366

Libro Frankenstein
Autor Mary Shelley
Total de oraciones 3,122
Total de tokens 75,303

Libro Dracula
Autor Bram Stoker
Total de oraciones 7,839
Total de tokens 162,771


In [6]:
#seleccionamos una muestra aleatoria de 100 oraciones por novela

SEED = 42
random.seed(SEED)

TAMANO_MUESTRA = 100

for titulo, datos in novelas_procesadas.items():
    if len(datos["oraciones"]) < TAMANO_MUESTRA:
        raise ValueError(f"{titulo} no tiene suficientes oraciones para crear la muestra")

    datos["muestra"] = random.sample(
        datos["oraciones"],
        TAMANO_MUESTRA
    )

    datos["tokens_muestra"] = sum(
        len(oracion)
        for oracion in datos["muestra"]
    )
    
    #mostramos un resumen de la muestra

    print(f"\nMuestra de {titulo}")
    print(f"Oraciones seleccionadas {len(datos['muestra']):,}")
    print(f"Tokens en la muestra {datos['tokens_muestra']:,}")
    print("Ejemplo de oracion de la muestra")
    print(datos["muestra"][0])


Muestra de Pride and Prejudice
Oraciones seleccionadas 100
Tokens en la muestra 1,980
Ejemplo de oracion de la muestra
['After', 'this', 'day', 'Jane', 'said', 'no', 'more', 'of', 'her', 'indifference']

Muestra de Frankenstein
Oraciones seleccionadas 100
Tokens en la muestra 2,309
Ejemplo de oracion de la muestra
['I', 'was', 'partly', 'urged', 'by', 'curiosity', 'and', 'compassion', 'confirmed', 'my', 'resolution']

Muestra de Dracula
Oraciones seleccionadas 100
Tokens en la muestra 1,731
Ejemplo de oracion de la muestra
['I', 'shall', 'wire', 'to', 'my', 'people', 'to', 'have', 'horses', 'and', 'carriages', 'where', 'they', 'will', 'be', 'most', 'convenient', 'Look', 'here', 'old', 'fellow', 'said', 'Morris', 'it', 'is', 'a', 'capital', 'idea', 'to', 'have', 'all', 'ready', 'in', 'case', 'we', 'want', 'to', 'go', 'horsebacking', 'but', 'don', 't', 'you', 'think', 'that', 'one', 'of', 'your', 'snappy', 'carriages', 'with', 'its', 'heraldic', 'adornments', 'in', 'a', 'byway', 'of', '

In [7]:
#construimos la tabla comparativa de libros completos y muestras

tabla_muestras = []

for titulo, datos in novelas_procesadas.items():
    tabla_muestras.append({
        "libro": titulo,
        "oraciones_totales": len(datos["oraciones"]),
        "tokens_totales": sum(len(oracion) for oracion in datos["oraciones"]),
        "oraciones_muestra": len(datos["muestra"]),
        "tokens_muestra": datos["tokens_muestra"]
    })

tabla_muestras = pd.DataFrame(tabla_muestras)

display(tabla_muestras)

print("\nResumen de la preparacion de la muestra")

for _, fila in tabla_muestras.iterrows():
    print(
        f"{fila['libro']} | "
        f"oraciones totales {fila['oraciones_totales']:,} | "
        f"tokens totales {fila['tokens_totales']:,} | "
        f"oraciones muestra {fila['oraciones_muestra']:,} | "
        f"tokens muestra {fila['tokens_muestra']:,}"
    )

,libro,oraciones_totales,tokens_totales,oraciones_muestra,tokens_muestra
0,Pride and Prejudice,5943,128366,100,1980
1,Frankenstein,3122,75303,100,2309
2,Dracula,7839,162771,100,1731



Resumen de la preparacion de la muestra
Pride and Prejudice | oraciones totales 5,943 | tokens totales 128,366 | oraciones muestra 100 | tokens muestra 1,980
Frankenstein | oraciones totales 3,122 | tokens totales 75,303 | oraciones muestra 100 | tokens muestra 2,309
Dracula | oraciones totales 7,839 | tokens totales 162,771 | oraciones muestra 100 | tokens muestra 1,731
